# 🩺 Prediccion de riesgo de diabetes (Mexico)

Proyecto de salud publica orientado a clasificar el riesgo de diabetes
(bajo, moderado, alto) usando variables clinicas y demograficas.
El modelo se plantea como herramienta preventiva y explicable.

## 0. Importar librerias y configuracion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')


## 1. Carga y exploracion inicial de datos

In [ ]:
DATA_PATH = 'data/Diabetes_Mexico.csv'
df = pd.read_csv(DATA_PATH)

print('Dimensiones:', df.shape)
display(df.head())
df.info()


## 2. Calidad de datos: faltantes y valores atipicos

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:
key_vars = ['edad', 'imc', 'glu_suero', 'hb1ac', 'insulina']
key_vars = [v for v in key_vars if v in df.columns]

def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_counts = {col: iqr_outliers(df[col].dropna()) for col in key_vars}
pd.Series(outlier_counts, name='outliers')


## 3. Analisis exploratorio (EDA)

In [ ]:
target = 'riesgo_diabetes_cat'

for col in key_vars:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x=target, y=col)
    plt.title(f'{col} por nivel de riesgo')
    plt.xlabel('Riesgo')
    plt.ylabel(col)
    plt.show()


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != target]

plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, cmap='viridis', center=0, linewidths=0.5)
plt.title('Correlacion entre variables numericas')
plt.show()


## 4. Seleccion y preparacion de variables

In [ ]:
features = df.drop(columns=[target], errors='ignore')
y = df[target].copy()

categorical_cols = [c for c in features.columns if features[c].dtype == 'object']
numeric_cols = [c for c in features.columns if c not in categorical_cols]

X_train, X_valid, y_train, y_valid = train_test_split(
    features,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)


## 5. Modelos: baseline y modelo avanzado

In [ ]:
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted'
    )
    print(f'\n{name}')
    print(f'Accuracy: {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')
    print('\nReporte por clase:')
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title(f'Matriz de confusion - {name}')
    plt.show()


In [ ]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    multi_class='multinomial',
    random_state=RANDOM_STATE
)

log_reg_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', log_reg)
])

log_reg_model.fit(X_train, y_train)
log_reg_pred = log_reg_model.predict(X_valid)
evaluate_model('Logistic Regression (baseline)', y_valid, log_reg_pred)


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', rf)
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_valid)
evaluate_model('Random Forest', y_valid, rf_pred)


## 6. ROC-AUC One-vs-Rest (si aplica)

In [ ]:
classes = np.sort(y_valid.unique())
y_valid_bin = label_binarize(y_valid, classes=classes)

if hasattr(rf_model.named_steps['model'], 'predict_proba'):
    y_score = rf_model.predict_proba(X_valid)
    roc_auc = roc_auc_score(y_valid_bin, y_score, average='weighted', multi_class='ovr')
    print(f'ROC-AUC OVR (Random Forest): {roc_auc:.4f}')


## 7. Interpretabilidad: importancia de variables

In [ ]:
def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = []
    if numeric_cols:
        feature_names.extend(numeric_cols)

    if categorical_cols:
        ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
        ohe_names = ohe.get_feature_names_out(categorical_cols)
        feature_names.extend(ohe_names.tolist())
    return feature_names

rf_fitted = rf_model.named_steps['model']
feature_names = get_feature_names(preprocess, numeric_cols, categorical_cols)
importances = pd.Series(rf_fitted.feature_importances_, index=feature_names)
top_importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
top_importances.sort_values().plot(kind='barh')
plt.title('Variables mas influyentes (Random Forest)')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

top_importances


## 8. Conclusiones, limitaciones y mejoras

**Conclusiones**
- El modelo Random Forest ofrece mejor rendimiento que el baseline.
- Variables metabolicas como glucosa, hb1ac e imc tienden a dominar la prediccion.

**Limitaciones**
- Posible sesgo por desbalance de clases.
- Falta de variables longitudinales para evaluar evolucion.

**Mejoras futuras**
- Probar Gradient Boosting y calibracion de probabilidades.
- Analisis de interpretabilidad avanzada (SHAP).

**Aplicaciones**
- Apoyo preventivo en salud publica.
- Priorizacion de pacientes para monitoreo.